In [ ]:
# =========================================================
# 02_historical_chart_ingestion.ipynb
# =========================================================

import requests
import pandas as pd
import time

from datetime import datetime, timedelta


# =========================================================
# CONFIG
# =========================================================

URL = "https://charts.youtube.com/youtubei/v1/browse?alt=json"

HEADERS = {
    "Content-Type": "application/json"
}


# =========================================================
# GENERATE WEEKLY DATES
# =========================================================

start_date = datetime(2021, 1, 7)
end_date = datetime(2026, 5, 14)

dates = []

current = start_date

while current <= end_date:

    dates.append(current.strftime("%Y%m%d"))

    current += timedelta(days=7)

print(f"Total weeks: {len(dates)}")


# =========================================================
# FIND trackViews RECURSIVELY
# =========================================================

def find_trackviews(obj):

    if isinstance(obj, dict):

        if "trackViews" in obj:
            return obj["trackViews"]

        for value in obj.values():

            result = find_trackviews(value)

            if result:
                return result

    elif isinstance(obj, list):

        for item in obj:

            result = find_trackviews(item)

            if result:
                return result

    return None


# =========================================================
# FETCH SINGLE WEEK
# =========================================================

def fetch_chart(chart_date):

    payload = {
        "context": {
            "client": {
                "clientName": "WEB_MUSIC_ANALYTICS",
                "clientVersion": "2.0",
                "hl": "en-GB",
                "gl": "IN",
                "theme": "MUSIC"
            }
        },

        "browseId": "FEmusic_analytics_charts_home",

        "query": (
            "perspective=CHART_DETAILS"
            "&chart_params_country_code=in"
            "&chart_params_chart_type=TRACKS"
            "&chart_params_period_type=WEEKLY"
            f"&chart_params_end_date={chart_date}"
        )
    }

    response = requests.post(
        URL,
        headers=HEADERS,
        json=payload
    )

    data = response.json()

    tracks = find_trackviews(data)

    if tracks is None:
        print(f"No tracks found for {chart_date}")
        return pd.DataFrame()

    rows = []

    for track in tracks:

        artists = ", ".join(
            [a["name"] for a in track.get("artists", [])]
        )

        rank = (
            track.get("chartEntryMetadata", {})
            .get("currentPosition")
        )

        previous_rank = (
            track.get("chartEntryMetadata", {})
            .get("previousPosition")
        )

        weeks_on_chart = (
            track.get("chartEntryMetadata", {})
            .get("periodsOnChart")
        )

        percent_change = (
            track.get("chartEntryMetadata", {})
            .get("percentViewsChange")
        )

        rows.append({

            "week": chart_date,

            "rank": rank,

            "previous_rank": previous_rank,

            "song": track.get("name"),

            "artists": artists,

            "views": int(track.get("viewCount", 0)),

            "video_id": track.get("encryptedVideoId"),

            "weeks_on_chart": weeks_on_chart,

            "percent_views_change": percent_change

        })

    df = pd.DataFrame(rows)

    df = df.sort_values("rank")

    return df


# =========================================================
# FETCH ALL WEEKS
# =========================================================

all_dfs = []

for d in dates:

    print(f"Fetching {d}")

    try:

        df = fetch_chart(d)

        if not df.empty:

            all_dfs.append(df)

            print(f"Success: {len(df)} tracks")

        else:

            print("Empty dataframe")

    except Exception as e:

        print(f"ERROR for {d}: {e}")

    # polite delay
    time.sleep(1)


# =========================================================
# COMBINE EVERYTHING
# =========================================================

master_df = pd.concat(all_dfs, ignore_index=True)

print("\nFINAL DATASET")
print(master_df.head())

print("\nShape:")
print(master_df.shape)


# =========================================================
# SAVE MASTER DATASET
# =========================================================

master_df.to_csv(
    "india_youtube_weekly_charts_2021_2026.csv",
    index=False
)

print("\nSaved:")
print("india_youtube_weekly_charts_2021_2026.csv")